# Prepare and Define

## init data

### datasets

In [5]:
import pandas as pd

strongs_df = pd.read_csv("strongs.csv")
lv65_df = pd.read_csv("latvian_full65.csv")
strongs_g = strongs_df.groupby(["book", "chapter", "verse"])
lv_g = lv65_df.groupby(["book", "chapter", "verse"])

### morphology definition

In [6]:
POS_MAP = {
    "V": "Verb",
    "N": "Noun",
    "Adv": "Adverb",
    "Adj": "Adjective",
    "Art": "Article",
    "DPro": "Demonstrative Pronoun",
    "IPro": "Interrogative / Indefinite Pronoun",
    "PPro": "Personal / Possessive Pronoun",
    "RecPro": "Reciprocal Pronoun",
    "RelPro": "Relative Pronoun",
    "RefPro": "Reflexive Pronoun",
    "Prep": "Preposition",
    "Conj": "Conjunction",
    "I": "Interjection",
    "Prtcl": "Particle",
    "Heb": "Hebrew Word",
    "Aram": "Aramaic Word"
}

PERSON_MAP = {"1": "1st Person", "2": "2nd Person", "3": "3rd Person"}
TENSE_MAP = {"P": "Present", "I": "Imperfect", "F": "Future",
             "A": "Aorist", "R": "Perfect", "L": "Pluperfect"}
MOOD_MAP = {"I": "Indicative", "M": "Imperative",
            "S": "Subjunctive", "O": "Optative",
            "N": "Infinitive", "P": "Participle"}
VOICE_MAP = {"A": "Active", "M": "Middle", "P": "Passive", "M/P": "Middle or Passive"}
CASE_MAP = {"N": "Nominative", "V": "Vocative", "A": "Accusative",
            "G": "Genitive", "D": "Dative"}
NUMBER_MAP = {"S": "Singular", "P": "Plural"}
GENDER_MAP = {"M": "Masculine", "F": "Feminine", "N": "Neuter"}
COMPARISON_MAP = {"C": "Comparative", "S": "Superlative"}

In [7]:
def parse_morph_code(code):
    if not isinstance(code, str) or not code:
        return {}

    parts = code.split("-")
    pos_key = parts[0]
    
    result = {"part_of_speech": POS_MAP.get(pos_key, pos_key)}

    if len(parts) < 2:
        return result

    # --- VERB LOGIC ---
    if pos_key == "V":
        tmv = parts[1]
        if len(tmv) >= 3:
            result["tense"] = TENSE_MAP.get(tmv[0])
            result["mood"] = MOOD_MAP.get(tmv[1])
            result["voice"] = VOICE_MAP.get(tmv[2])
        
        if len(parts) >= 3:
            third_part = parts[2]
            # FIX: Participles have Case-Gender-Number, not Person-Number
            if result.get("mood") == "Participle":
                for char in third_part:
                    if char in CASE_MAP:
                        result["case"] = CASE_MAP.get(char)
                    elif char in GENDER_MAP:
                        result["gender"] = GENDER_MAP.get(char)
                    elif char in NUMBER_MAP:
                        result["number"] = NUMBER_MAP.get(char)
            else:
                # Finite verbs: Person-Number
                if len(third_part) >= 2:
                    result["person"] = PERSON_MAP.get(third_part[0])
                    result["number"] = NUMBER_MAP.get(third_part[1])
        return result

    # --- NOUN / ADJ / ART / PRONOUN LOGIC ---
    details = parts[1]
    for char in details:
        if char in CASE_MAP:
            result["case"] = CASE_MAP.get(char)
        elif char in GENDER_MAP:
            result["gender"] = GENDER_MAP.get(char)
        elif char in NUMBER_MAP:
            result["number"] = NUMBER_MAP.get(char)
        elif char in COMPARISON_MAP:
            result["comparison"] = COMPARISON_MAP.get(char)
        elif char in PERSON_MAP and "Pro" in pos_key:
            result["person"] = PERSON_MAP.get(char)

    return result

In [8]:
def build_token_object(row, latvian_words):

    morph_code = row["en_title"]   # your morphology column

    return {
        "greek_form": row["form"],
        "latvian_words": latvian_words,
        "strong_number": row["strong_num"],
        "strong_title": row["strong_title"],
        "morph_code": morph_code,
        "morph_parsed": parse_morph_code(morph_code)
    }

## build and render

### defs

In [9]:
#print(strongs_df.head())
#print(lv65_df.head())

In [10]:
import pandas as pd
import time
from pathlib import Path
import json

def build_verse_object(book, chapter, verse, strongs_g, lv_g):

    key = (book, chapter, verse)

    if key not in strongs_g.groups:
        return None

    if key not in lv_g.groups:
        return None

    strongs_verse = strongs_g.get_group(key)

    latvian_text = lv_g.get_group(key).iloc[0]["text"]
    #print(strongs_verse.sort_values("word"))
    strong_sorted = strongs_verse.sort_values("word")
    strong_data_list = strong_sorted.to_dict("records")
    #print (strong_data_list)
    #return
    greek_text = " ".join(
        strong_sorted["form"].astype(str)
    )
    #greek_text = " ".join([t["form"] for _, t in strongs_verse.sort_values("word").iterrows()])
    
    posfile_path = Path("bible") / str(book) / str(chapter) / f"{book}_{chapter}_{verse}.txt"

    mappings=[]
    leftover_latvian = []
    if posfile_path.exists():
        with open(posfile_path, "r", encoding="utf-8") as f:
            dobj = json.load(f)
        if(dobj.get("locus", None)):
            if(dobj["locus"] != f"{book}_{chapter}_{verse}"):
               raise Exception("does not match!" +" :"+ dobj["locus"] +" " +f"{book}_{chapter}_{verse}")
        if len(strong_sorted) != len(dobj["greek_words"]):
            raise Exception("words lost or excess! at" +" :"+ f"{(book, chapter, verse)}" + "\n db: " + \
                            str(len(strong_sorted)) + " act: " + str(len(dobj["greek_words"])))
        for i in range(len(strong_sorted)):
            gw = dobj['greek_words'][i] #search by index match
            strong_row = strong_data_list[i]
            if int(gw['index']) != i or int(gw['index']) != int(strong_row['word']):
                print(dobj)
                raise Exception("wrong index! Try running python renumber.py ." +" :"+ f"{(book, chapter, verse)}" + " " + str(gw["index"]) + " index should be: " + str(i) + " and in list: " + str(strong_row['word']))
            gw.update({
                "strong_num": strong_row.get("strong_num"),
                "form": strong_row.get("form"),
                "translit_title": strong_row.get("translit_title"),
                "translit": strong_row.get("translit"),
                "strong_en_title": strong_row.get("strong_en_title")
            })
            
            mappings.append(gw)
        leftover_latvian = dobj.get("leftover_latvian", [])
    else:
        print("File does not exist:", posfile_path)

    return {
        "book": book,
        "chapter": chapter,
        "verse": verse,
        "greek_text": greek_text,
        "latvian_text": latvian_text,
        "mappings": mappings,
        "leftover_latvian": leftover_latvian
    }


def process_book_full(book, strongs_g, lv_g, exclude_list=[], startatchap=1, stopatchap=-1):
    results = []
    total_start = time.perf_counter()   # ⏳ start total timer
    # Drive iteration from Latvian (verse-level dataset)
    keyss = sorted(lv_g.groups.keys())
    for i, key in enumerate(keyss):
        b, chapter, verse = key
        if b != book:
            continue
        
        if key in exclude_list:
            continue

        #current chunk limit
        if stopatchap>0 and chapter >= stopatchap:
            break

        if startatchap>0 and chapter < startatchap:
            continue

        result = build_verse_object(b, chapter, verse, strongs_g, lv_g)

        results.append(result)

        #if len(results) >30:
        #    break

    return results



In [11]:
#print(data_to_html_render(data))
#with open(f"{Path("bible") / str(b) / f'{chapter}.html'}", "w", encoding="utf-8") as f:
#take data as filter to generate only those books and their chaps inside
def render_fragments_from_data(data):
    if not data: return
    books = {}
    for n, i in enumerate(data):
        bk = i.get("book", None)
        ch = i.get("chapter", None)
        if not bk or not ch:
            continue
        if not books.get(bk, None):
            books[bk] = {}
        if not books[bk].get(ch, None):
            books[bk][ch] = []
        #sorted would be nice wouldnt it with a lambda return i.get("verse", -1) but it seems to be sorted already..
        books[bk][ch].append(i)
    
    for bk, v in books.items():
        for ch, vv in v.items():
            with open(Path("bible") / str(bk) / f"{ch}.html", "w", encoding="utf-8") as f:
                f.write(chapter_to_html_render(vv))

### template style def

In [17]:
strongs_p_dir="strongs_p_g"
def make_audio_players(strong_num_raw, v_num, word_idx, strongs_p_dir="strongs_p"):
    """
    Returns HTML for audio+button pairs for a given strong number.
    Outputs one pair for the base file (h0001.mp3) and one per variant (-2, -3...).
    Returns empty string if strong number is invalid (None, '-1', '0', negative).
    """
    if not strong_num_raw:
        return ""
    try:
        sn = int(strong_num_raw)
    except (ValueError, TypeError):
        return ""
    if sn <= 0:
        return ""

    skey = f"g{sn:04d}"
    base_path = Path(strongs_p_dir) / f"{skey}.mp3"

    # collect all variant files that exist: g0001.mp3, g0001-2.mp3, g0001-3.mp3 ...
    variants = []
    if base_path.exists():
        variants.append((f"/{strongs_p_dir}/{skey}.mp3", ""))   # label empty for first
    vi = 2
    while True:
        vpath = Path(strongs_p_dir) / f"{skey}-{vi}.mp3"
        if not vpath.exists():
            break
        variants.append((f"/{strongs_p_dir}/{skey}-{vi}.mp3", f" {vi}"))
        vi += 1

    if not variants:
        return ""

    out = "<br>"
    for src, label in variants:
        uid = f"{skey}_v{v_num}_w{word_idx}{('_' + str(vi)) if label else ''}"
        # make uid truly unique per variant
        uid = f"{skey}_v{v_num}_w{word_idx}{label.strip()}"
        out += (
            f'<audio id="aud_{uid}" src="{src}" '
            f'onended="document.getElementById(\'btn_{uid}\').textContent=\'▶{label}\'"></audio>'
            f'<button id="btn_{uid}" style="font-size:0.8em;padding:1px 5px;cursor:pointer" '
            f'onclick="var a=document.getElementById(\'aud_{uid}\');'
            f'if(a.paused){{a.play();this.textContent=\'⏹{label}\';}}else{{a.pause();a.currentTime=0;this.textContent=\'▶{label}\';}}">'
            f'▶{label}</button> '
        )
    return out
    
def chapter_to_html_render(data):
    if not data or len(data) < 1: 
        return ""

    # CSS updated with tooltip support and a cleaner table look
    css = """
    <style>
        body { font-family: 'Segoe UI', Tahoma, sans-serif; line-height: 1.6; color: #333; max-width: 1600px; margin: 0 auto; padding: 20px; background-color: #f8f9fa; }
        h1 { text-align: center; color: #2c3e50; border-bottom: 2px solid #3498db; padding-bottom: 10px; }
        .verse-container { background-color: white; border-radius: 10px; box-shadow: 0 4px 8px rgba(0,0,0,0.1); margin-bottom: 40px; padding: 25px; border-left: 5px solid #3498db; }
        .verse-header { font-weight: bold; color: #2c3e50; background-color: #ecf0f1; padding: 8px 15px; border-radius: 20px; margin-bottom: 15px; }
        .verse-lines { display: flex; flex-wrap: wrap; gap: 15px; margin-bottom: 20px; }
        .line-box { flex: 1 1 45%; min-width: 300px; }
        .line-label { font-weight: bold; color: #16a085; margin-bottom: 5px; }
        .line-content { background-color: #f0f7f4; padding: 12px; border-radius: 8px; border: 1px solid #bdc3c7; font-size: 1.1em; }
        .greek-line { background-color: #f0f0f0; font-family: 'Times New Roman', serif; }
        
        /* Table Styles */
        .mapping-table { width: 100%; border-collapse: collapse; margin-top: 15px; font-size: 0.9em; }
        .mapping-table th { background-color: #3498db; color: white; padding: 12px; text-align: left; position: sticky; top: 0; }
        .mapping-table td { padding: 10px; border-bottom: 1px solid #ddd; vertical-align: top; }
        
        .greek-word { font-weight: bold; color: #8e44ad; font-size: 1.1em; }
        .greek-form { color: #7f8c8d; font-weight: normal; font-size: 0.85em; }
        .latvian-word { font-weight: bold; color: #27ae60; }
        
        /* Morphology Tooltip Style */
        .morph-info { 
            font-style: italic; 
            color: #e67e22; 
            cursor: text; 
            border-bottom: 1px dotted #e67e22; 
            display: inline-block;
        }
        .definition-cell { color: #555; font-size: 0.85em; line-height: 1.3; max-width: 400px; }
        .index-badge { display: inline-block; width: 25px; height: 25px; background-color: #e74c3c; color: white; border-radius: 50%; text-align: center; line-height: 25px; margin-right: 8px; }
    </style>
    """

    book_title = data[0].get('book', 'Bible').capitalize()
    chapter = data[0].get('chapter', '')
    html = f"<!DOCTYPE html><html><head><meta charset='UTF-8'>{css}</head><body>"
    html += f"<h1>📖 {book_title} Chapter {chapter}</h1>"

    for verse_data in data:
        v_num = verse_data.get('verse')
        locus = f"{book_title} {chapter}:{v_num}"
        
        html += f'<div class="verse-container">'
        html += f'<div class="verse-header"><span class="index-badge">{v_num}</span> {locus}</div>'
        
        # ... (text lines remain the same) ...
        html += f'''
        <div class="verse-lines">
            <div class="line-box">
                <div class="line-label">🇬🇷 Greek:</div>
                <div class="line-content greek-line">{verse_data.get('greek_text', '')}</div>
            </div>
            <div class="line-box">
                <div class="line-label">🇱🇻 Latvian:</div>
                <div class="line-content">{verse_data.get('latvian_text', '')}</div>
            </div>
        </div>
        '''

        html += '''<table class="mapping-table"><thead><tr>
                    <th>Greek (Form)</th><th>Latvian</th><th>Strong's</th><th>Morphology</th><th>Definition</th>
                </tr></thead><tbody>'''
        for m_idx, m in enumerate(verse_data.get('mappings', [])):
            lv_words = ", ".join(m.get('latvian', [])) if m.get('latvian') else "-"
            strong = f"G{m.get('strong_num')}" if m.get('strong_num') else "-"
            
            # 1. Parse the morphology for the tooltip
            raw_morph = m.get('strong_en_title', '')
            morph_dict = parse_morph_code(raw_morph)
            # Create a comma-separated string of the full names
            #full_desc = ", ".join([f"{k}: {v}" for k, v in morph_dict.items() if v])
            full_desc = ", ".join([f"{k.replace('_', ' ').lower()}: {v.title()}" for k, v in morph_dict.items() if v])
            audio_html = make_audio_players(m.get('strong_num'), v_num, m_idx)
            #print(m.get('strong_num'))

            #form ir no strongs dataseta, kamēr greek ir no LLM atbildes citāts; lai gan jāsakrīt, tomēr labāk sets.
            #morph saite pagaidām placeholders no blueletterbible
            html += f'''
                <tr>
                    <td class="greek-word">
                        {m.get('form', '')} <span class="greek-form">({m.get('translit', '')})</span>
                        {audio_html}
                    </td>
                    <td class="latvian-word">{lv_words}</td>
                    <td><a href="https://www.blueletterbible.org/lexicon/{strong.lower()}/" target="_blank">{strong}</a></td>
                    <td>
                        <span class="morph-info" title="{full_desc}">{raw_morph}</span>
                    </td>
                    <td class="definition-cell">{m.get('translit_title', '')}</td>
                </tr>
            '''
        if(len(verse_data.get('leftover_latvian', []))>0):
            html += f'''
                <tr>
                    <td>
                    <span class="greek-form">- (no match)</span>
                    </td>
                    <td colspan="4">
                        {" ,".join(verse_data.get('leftover_latvian', []))}
                    </td>
                </tr>
            '''
        html += "</tbody></table></div>"

    html += "</body></html>"
    return html

# doit

### done1

In [270]:
ex_list = []
ex_list = [("john", 1, i) for i in range (1, 51+1)] # if i != 42]
ex_list.extend([("john", 2, i) for i in range (1, 25+1)])
ex_list.extend([("john", 3, i) for i in range (1, 36+1)])
ex_list.extend([("john", 4, i) for i in range (1, 54+1)])
ex_list.extend([("john", 5, i) for i in range (1, 47+1)])
ex_list.extend([("john", 6, i) for i in range (1, 71+1)])
ex_list.extend([("john", 7, i) for i in range (1, 53+1)])
ex_list.extend([("john", 8, i) for i in range (1, 59+1)])
#ex_list.extend([("john", 12, 38)])
#ex_list.extend([("john", 12, 49)])
#ex_list.extend([("john", 15, 23)])
#ex_list.extend([("john", 18, 2)])
#ex_list.extend([("john", 18, 40)])
data = process_book_full(
    "john",
    strongs_g,
    lv_g,
    ex_list,
    -1,
    -1
)
render_fragments_from_data(data)

In [467]:
curchap = -2
data = process_book_full(
    "mark",
    strongs_g,
    lv_g,
    ex_list,
    curchap,
    curchap+1
)
# 8 9
render_fragments_from_data(data)

In [447]:
curchap = -2
data = process_book_full(
    "matthew",
    strongs_g,
    lv_g,
    ex_list,
    curchap,
    curchap+1
)
render_fragments_from_data(data)

In [ ]:
curchap = 24
data = process_book_full(
    "luke",
    strongs_g,
    lv_g,
    ex_list,
    curchap,
    curchap+1
)

render_fragments_from_data(data)
print("done")

In [512]:

curchap = -2
data = process_book_full(
    "1_corinthians",
    strongs_g,
    lv_g,
    ex_list,
    curchap,
    curchap+1
)

render_fragments_from_data(data)
print("done")

done


In [ ]:
curchap = len("123456789*123")
data = process_book_full(
    "2_corinthians",
    strongs_g,
    lv_g,
    ex_list,
    curchap,
    curchap+1
)

render_fragments_from_data(data)
print("done")

In [ ]:
curchap = len("12345")
data = process_book_full(
    "1_john",
    strongs_g,
    lv_g,
    ex_list,
    curchap,
    curchap+1
)

render_fragments_from_data(data)
print("done")

### undone 2

In [538]:
curchap = len("12345")
data = process_book_full(
    "1_peter",
    strongs_g,
    lv_g,
    ex_list,
    curchap,
    curchap+1
)

render_fragments_from_data(data)
print("done")

done


In [18]:
curchap = len("12345")
data = process_book_full(
    "acts",
    strongs_g,
    lv_g,
    [],
    curchap,
    curchap+1
)
render_fragments_from_data(data)
print("done")

435
1161
5100
367
3686
4862
4551
3588
1135
846
4453
2933
2532
3557
575
3588
5092
4894
2532
3588
1135
2532
5342
3313
5100
3844
3588
4228
3588
652
5087
2036
1161
3588
4074
367
1223
5101
4137
3588
4567
3588
2588
4771
5574
4771
3588
4151
3588
40
2532
3557
575
3588
5092
3588
5564
3780
3306
4771
3306
2532
4097
1722
3588
4674
1849
5225
5101
3754
5087
1722
3588
2588
4771
3588
4229
3778
3756
5574
444
235
3588
2316
191
1161
3588
367
3588
3056
3778
4098
1634
2532
1096
5401
3173
1909
3956
3588
191
450
1161
3588
3501
4958
846
2532
1627
2290
1096
1161
5613
5610
5140
1292
2532
3588
1135
846
3361
1492
3588
1096
1525
611
1161
4314
846
4074
2036
1473
1487
5118
3588
5564
591
3588
1161
2036
3483
5118
3588
1161
4074
4314
846
5101
3754
4856
4771
3985
3588
4151
2962
2400
3588
4228
3588
2290
3588
435
4771
1909
3588
2374
2532
1627
4771
4098
1161
3916
4314
3588
4228
846
2532
1634
1525
1161
3588
3495
2147
846
3498
2532
1627
2290
4314
3588
435
846
2532
1096
5401
3173
1909
3650
3588
1577
2532
1909
3956
3588
191
37